# Lesson 05 - Agentic RAG

## Setup

This notebook demonstrates the Agentic RAG (Retrieval-Augmented Generation) pattern using the Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint, for example `https://azure-search-service-01.search.windows.net`
- `AZURE_SEARCH_INDEX_NAME` — your Azure AI Search index name, for example `demo-datasource-ks-index`
- `AZURE_SEARCH_API_KEY` — your Azure AI Search query or admin API key
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)

In [1]:
%pip install agent-framework azure-ai-projects azure-identity azure-search-documents python-dotenv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
search_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
search_index_name = os.getenv("AZURE_SEARCH_INDEX_NAME", "demo-datasource-ks-index")
search_api_key = os.getenv("AZURE_SEARCH_API_KEY")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name,
    "AZURE_SEARCH_SERVICE_ENDPOINT": search_endpoint,
    "AZURE_SEARCH_API_KEY": search_api_key,
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [4]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

In [5]:
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=search_index_name,
    credential=AzureKeyCredential(search_api_key),
)

print(f"Azure AI Search index configured: {search_index_name}")

Azure AI Search index configured: demo-datasource-ks-index


## What is Agentic RAG?

Traditional RAG follows a fixed pipeline: retrieve documents, then generate a response. **Agentic RAG** goes further by giving the agent autonomy to decide **when** and **how** to retrieve information.

With Agentic RAG, the agent can:
- **Decide** whether retrieval is needed before answering a question
- **Choose** which data source or tool to query
- **Evaluate** retrieved results and perform follow-up retrievals if the first attempt is insufficient
- **Combine** information from multiple retrieval steps into a coherent answer

This makes the agent more flexible and accurate compared to a static retrieve-then-generate pipeline.

## Creating a Search Tool

In Agentic RAG, external data sources are wrapped as **tools** that the agent can invoke on demand. This lets the agent treat retrieval as just another action it can take, rather than a mandatory step.

Below we expose your Azure AI Search index as a tool the agent can call to look up destination information.

In [8]:
def format_search_result(result: dict) -> str:
    visible_fields = []

    for field_name, field_value in result.items():
        if field_name.startswith("@search."):
            continue
        if field_value is None:
            continue
        if field_name.lower().endswith("vector"):
            continue

        field_text = " ".join(str(field_value).split())
        if len(field_text) > 500:
            field_text = f"{field_text[:500]}..."
        visible_fields.append(f"{field_name}: {field_text}")

    return "\n".join(visible_fields)


@tool(approval_mode="never_require")
def search_knowledge_base(
    query: Annotated[str, "The search query for the Azure AI Search index"]
) -> str:
    """Search the Azure AI Search index for relevant information."""
    results = search_client.search(search_text=query, top=3)
    matches = []

    for result in results:
        formatted_result = format_search_result(dict(result))
        if formatted_result:
            matches.append(formatted_result)

    return (
        "\n\n---\n\n".join(matches)
        if matches
        else "No matching documents found in the Azure AI Search index."
    )

In [9]:
sample_results = search_client.search(search_text="architecture", top=1)

for result in sample_results:
    print(format_search_result(dict(result)))

snippet_parent_id: aHR0cHM6Ly9zdGFjdGRlbW8yMDI2MDYyNS5ibG9iLmNvcmUud2luZG93cy5uZXQvZGVtby1kYXRhc291cmNlL0JBUkJJRSUyMEZJTkFMJTIwMjAyMyUyMEdHJTIwU0NSRUVOUExBWS5wZGY1
blob_url: https://stactdemo20260625.blob.core.windows.net/demo-datasource/BARBIE%20FINAL%202023%20GG%20SCREENPLAY.pdf
uid: ac14eb9d3111_aHR0cHM6Ly9zdGFjdGRlbW8yMDI2MDYyNS5ibG9iLmNvcmUud2luZG93cy5uZXQvZGVtby1kYXRhc291cmNlL0JBUkJJRSUyMEZJTkFMJTIwMjAyMyUyMEdHJTIwU0NSRUVOUExBWS5wZGY1_pages_59
snippet: was cutting a Ken’s steak for him...? GLORIA Welcome back, Madame President. BARBIE MARGOT (V.O.) And then we’ll recruit the now unbrainwashed Barbies to our cause. They can be the new decoys. INTERCUT THE PLAN. The Barbies distract the Kens by pretending to be helpless and then Gloria deprograms them. GLORIA (V.O.) Tell him you’ve never seen the Godfather and you’d love him to explain it to you. In a Ken Mojo Dojo Casa House, Ken Kingsley sits with Barbie Sharon in front of one of the giant TVs...


## Building the RAG Agent

Now we create an agent that is instructed to **always retrieve information before answering**. The agent uses the `search_knowledge_base` tool to ground its responses in your Azure AI Search index rather than relying on its own training data.

In [10]:
agent = client.as_agent(
    tools=[search_knowledge_base],
    name="AzureSearchRAGAgent",
    instructions="""You are a knowledgeable assistant. Before answering questions:
1. ALWAYS search the Azure AI Search index first
2. Base your answers on retrieved information
3. If information is not in the index, say so clearly
4. Cite concrete details from the retrieved snippets.""",
)

response = await agent.run(
    "What does the knowledge base say about Barbie and Gloria?",
)
print(response)

The knowledge base reveals some interactions and roles of Barbie and Gloria in a screenplay context:

1. Gloria seems to be a strong character involved in a tense situation involving Mattel executives and other characters, with lines like "I am WIDE awake Sasha!" and taking decisive actions such as smashing into a median strip to evade or confront others (snippet from page 40).

2. Barbie is mentioned alongside Gloria in contexts suggesting they are involved in some kind of conflict or plan concerning "unbrainwashed Barbies" and Kens. Barbie Margot (a version or character of Barbie) is involved in recruiting and distracting, while Gloria is depicted as deprogramming them (snippet from page 59).

3. There is a scene described where Barbie, Gloria, and others are rollerblading towards "Barbie Land," which hints at a fantastical or unusual setting (snippet from page 42).

Overall, Barbie and Gloria appear to be central characters engaged in a narrative involving conflict, strategy, and ac

## Iterative Retrieval — The Maker-Checker Pattern

A key advantage of Agentic RAG is **iterative retrieval**. The agent can perform multiple rounds of search to verify, refine, or expand on its initial findings — similar to a "maker-checker" workflow:

1. **Maker step**: The agent retrieves initial information and drafts a response.
2. **Checker step**: The agent performs additional retrievals to verify details or fill gaps.

Below, the agent is asked a question that requires comparing information from multiple retrieval results, prompting it to search several times.

In [ ]:
checker_agent = client.as_agent(
    tools=[search_knowledge_base],
    name="AzureSearchRAGCheckerAgent",
    instructions="""You are a meticulous assistant who double-checks answers against retrieved sources.
When answering questions:
1. Search for relevant documents first
2. Search again with specific names, topics, or phrases from the first results
3. Compare the retrieved snippets
4. Present a final answer grounded in the index
5. If any detail seems incomplete, say what is missing.""",
)

response = await checker_agent.run(
    "Compare what the knowledge base says about Barbie and Ken.",
)
print(response)

Based on your $175/day budget and April travel timing, here are the options:

1. Tokyo: Costs around $200-250/day, slightly above your budget, but April is ideal for cherry blossoms. Highlights are Shibuya, temples, and sushi.

2. Paris: Costs about $180-250/day; also just above your budget. April is a great time to visit with fewer crowds. Known for Eiffel Tower, Louvre, and cuisine.

3. Barcelona: Costs about $150-200/day. April is a good time with mild weather. Highlights include Gaudí architecture, La Rambla, and beaches. Slightly flexible budget could work here.

4. Cape Town: Costs $100-150/day, fitting your budget well. However, April is less ideal as the best time is Nov-Mar. Attractions include Table Mountain, wine regions, and wildlife.

Recommendation:
- For strict budget adherence with decent timing: Cape Town fits budget but not ideal in April.
- For best timing and near budget: Barcelona is best, possibly stretching budget slightly.
- Tokyo and Paris are slightly above bu

## Summary

In this lesson you learned how to build an **Agentic RAG** system using the Microsoft Agent Framework:

- **Agentic RAG** lets agents autonomously decide when to retrieve information, making retrieval dynamic rather than fixed.
- **Tools as data sources**: External knowledge bases, like Azure AI Search indexes, are wrapped as tools the agent can invoke.
- **Iterative retrieval**: The maker-checker pattern enables the agent to perform multiple retrieval rounds — searching, verifying, and refining — before producing a final answer.

This pattern lets you replace static in-memory data with a real Azure AI Search index while keeping retrieval under the agent's control.